In [ ]:
"""
plot_gmm_overlap_score.ipynb
Author: Henry Yuan
Description: Code for Figure 6 plot. Plots overlap score based on Gaussian Mixture Model
classification error. Lower values indicate better separability of roundness stages for a given metric
"""

# imports
import numpy as np
import sys
import matplotlib.pyplot as plt
sys.path.append('../../src')
import pandas as pd
from format_plots import format_plots
import analyze_synthetics as asynth

CR, CS, CE, CL, CRD, CED, CWAD, CSTD = format_plots() # format plots and collect colors

In [ ]:
# load roundness data for synthetic grains
data_file = '../../data/synthetic/random_synthetics/synthetic_stats.csv'
synthetic_stats_df = pd.read_csv(data_file)

# unpack into original roundness stage (round_stages), vertex count (nvs), and desired roundness metrics
metric_names = ['std_curv','wadell']
round_stages = synthetic_stats_df['roundness']
nvs = synthetic_stats_df['vertex_count']
std_curv = synthetic_stats_df['std_curv']
wadell = synthetic_stats_df['wadell']

# format data
all_round_data = np.zeros((len(metric_names), 3, 1000))
vertex_counts = np.zeros((3,1000))

# loop through all roundness stages
for i,r in enumerate([0,0.5,1]):
    # subscript _i means "specific for this roundness stage"

    # get bool index list of all grains with roundness = r
    bool_list_i = (round_stages == r)
    # index into arrays to get subset of entire arrays for this specific r value
    vertex_counts[i] = nvs[bool_list_i]
    all_round_data[0,i] = wadell[bool_list_i]
    all_round_data[1,i] = std_curv[bool_list_i]

In [ ]:
# *** PLOT OVERLAP SCORE ***

regime_threshold = 5000 # threshold vertex count between "small mesh" and "large mesh" regimes

metric_labels = [r'$R_{W,\kappa}$',r'$R_{\sigma,\kappa}$']

ranges = np.array([[1000,10001],[1000,regime_threshold],[regime_threshold,10001]])
range_labels = ['full range\n(1,000 - 9,999)', 'small mesh regime\n(1,000 - 4,999)', 'large mesh regime\n(5,000 - 9,999)']
n_points = len(ranges)
error_rates = asynth.compute_gmm_overlap_score_in_ranges(all_round_data, vertex_counts, ranges) # these are the overlap scores

fig, ax = plt.subplots(1, 1, figsize=(4.9, 2))

x = np.arange(len(range_labels))
width = 0.20

group_half_width = ((len(metric_labels) - 1) / 2) * width + width / 2
side_margin = 0.25
ax.set_xlim(
    -group_half_width - side_margin,
    (len(range_labels) - 1) + group_half_width + side_margin
)

# set colors
colors = [CWAD, CSTD]
for i, metric in enumerate(metric_labels):
    ax.bar(
        x + (i - (len(metric_labels) - 1) / 2) * width,
        error_rates[i],
        width=width,
        color=colors[i],
        label=metric
    )

ax.set_xticks(x)
ax.set_xticklabels(range_labels)

ax.legend()
ax.set_ylabel('Overlap score, S')

plt.tight_layout()
plt.savefig('figure6.pdf')

